In [1]:
import pandas as pd

df = pd.read_csv("data/raw/reviews_raw.csv")
print("Rows and columns:", df.shape)
print(df.columns.tolist())
df.head()

Rows and columns: (63872, 11)
['reviewId', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'app', 'lang']


,reviewId,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,app,lang
0,571aca13-712b-4756-8682-fe6e88fb1ad2,ما يشتغل عندي بطلب من عندكم بس ما يشتغل لو ما ...,1,0,8.0.300,2026-10-02 15:05:02,"""نعتذر لك على التجربة غير المرضية، وحرصًا منا ...",2026-10-03 07:04:16,8.0.300,HungerStation,ar
1,82aa253b-b665-465b-85db-0e3a0eec4072,خدمة توصيل اللي في عشرة دقائق يجيك بعد 3ساعات ...,1,0,8.0.300,2026-10-02 14:37:41,"""نعتذر لك على التجربة غير المرضية، وحرصًا منا ...",2026-10-03 07:03:59,8.0.300,HungerStation,ar
2,563c08df-5c8e-4ca4-9739-4e6cf261b4da,جيد لكن هناك مبالغه في رسوم التوصيل,3,0,8.0.300,2026-10-02 11:50:51,NaN,NaN,8.0.300,HungerStation,ar
3,587821d4-470a-4cd7-81f8-def8b8481a65,اجمل برنامج على الاطلاق,5,0,8.0.300,2026-10-02 08:47:31,شكرا لك ، سعِدنا بتقييمك ، نسعى لتقديم الافضل ...,2026-10-03 07:01:05,8.0.300,HungerStation,ar
4,b00eaac0-4a10-43b0-afa7-8870eaf49f21,مفيد جدا بس التكاليف عاليه.,5,0,8.0.300,2026-10-02 08:35:43,شكرا لك ، سعِدنا بتقييمك ، نسعى لتقديم الافضل ...,2026-10-03 07:00:05,8.0.300,HungerStation,ar


In [2]:
# 1) How many missing values in each column?
print(df.isnull().sum())
print("-" * 40)

# 2) Date range for each app
df["at"] = pd.to_datetime(df["at"])
print(df.groupby("app")["at"].agg(["min", "max", "count"]))
print("-" * 40)

# 3) How long are the reviews? (number of characters)
df["length"] = df["content"].astype(str).str.len()
print(df["length"].describe())

reviewId                    0
content                     1
score                       0
thumbsUpCount               0
reviewCreatedVersion    12542
at                          0
replyContent            40531
repliedAt               40531
appVersion              12542
app                         0
lang                        0
dtype: int64
----------------------------------------
                              min                 max  count
app                                                         
HungerStation 2023-12-22 06:28:23 2026-10-02 15:05:02  18400
Jahez         2023-11-08 01:38:55 2026-10-02 14:50:51   6200
Keeta         2023-05-21 17:45:16 2026-10-02 15:40:00  22272
Mrsool        2023-12-27 21:46:53 2026-10-02 11:59:20  13000
ToYou         2023-12-26 17:18:56 2026-10-02 11:47:26   4000
----------------------------------------
count    63871.000000
mean        76.760157
std        102.602302
min          1.000000
25%         11.000000
50%         34.000000
75%         99.0

In [3]:
keeta = df[df["app"] == "Keeta"]

# How many Keeta reviews are BEFORE it entered Saudi (Sep 2024)?
before = keeta[keeta["at"] < "2024-09-01"]
print("Before Saudi launch:", len(before))
print(before["lang"].value_counts())

# Show 15 random English Keeta reviews AFTER Saudi launch
after_en = keeta[(keeta["at"] >= "2024-09-01") & (keeta["lang"] == "en")]
after_en[["at", "score", "content"]].sample(15)

Before Saudi launch: 163
lang
en    163
Name: count, dtype: int64


,at,score,content
43764,2025-09-01 00:14:23,1,the worst app in delivery time and there custo...
42129,2025-10-31 02:58:38,1,خدمه سيئه للغايه لا أنصح به أبدا
37266,2026-07-13 07:45:27,1,"o documento CNH expira no dia do vencimento, a..."
44546,2025-06-16 05:59:29,5,Good food
40359,2026-01-14 07:14:21,3,والله شي جدا جدا غريب ان مافي دفع عند الاستلام...
42882,2025-10-08 20:25:11,1,"Asked for my location initially, took my full ..."
45447,2025-03-03 05:23:29,5,Premium Services
38954,2026-03-28 17:55:25,1,no refund got scammed twice now because of them.
42527,2025-10-18 04:40:42,5,best experience with keeta delivery time good ...
38967,2026-03-28 05:27:16,1,"Keita Delivery Service Dubai is very slow, bad..."


In [4]:
import pandas as pd
import os

df = pd.read_csv("data/raw/reviews_raw.csv")
df["at"] = pd.to_datetime(df["at"])
print("Start:", len(df))

# 1) Remove empty reviews
df = df.dropna(subset=["content"])
print("After removing empty:", len(df))

# 2) Keep only reviews written in Arabic (check the text itself)
df = df[df["content"].str.contains(r"[\u0600-\u06FF]")]
print("After keeping Arabic only:", len(df))

# 3) Keep only 2024 onwards (same period for all apps)
df = df[df["at"] >= "2024-01-01"]
print("After keeping 2024+:", len(df))

# 4) Remove Keeta reviews before its Saudi launch
before_launch = (df["app"] == "Keeta") & (df["at"] < "2024-09-01")
df = df[~before_launch]
print("After removing Keeta pre-launch:", len(df))

# 5) Remove reviews that mention non-Saudi places or currencies
non_saudi = ["دبي", "الامارات", "الإمارات", "ابوظبي", "أبوظبي", "الشارقة",
             "الدوحة", "قطر", "الكويت", "البحرين", "المنامة", "مسقط",
             "درهم", "دينار"]
mentions = df["content"].str.contains("|".join(non_saudi))
print("Reviews mentioning non-Saudi places:", mentions.sum())
df = df[~mentions]
print("After removing non-Saudi mentions:", len(df))

# 6) Save the clean data
df = df.drop(columns=["lang"])
os.makedirs("data/clean", exist_ok=True)
df.to_csv("data/clean/reviews_clean.csv", index=False, encoding="utf-8-sig")
print("-" * 40)
print("Saved! Final count:", len(df))
print(df["app"].value_counts())

Start: 63872
After removing empty: 63871
After keeping Arabic only: 46630
After keeping 2024+: 46259
After removing Keeta pre-launch: 46259
Reviews mentioning non-Saudi places: 111
After removing non-Saudi mentions: 46148
----------------------------------------
Saved! Final count: 46148
app
HungerStation    13671
Keeta            12951
Mrsool           11395
Jahez             4819
ToYou             3312
Name: count, dtype: int64


In [5]:
import re

def normalize_arabic(text):
    text = str(text)
    text = re.sub(r"[إأآ]", "ا", text)            # أ إ آ  -> ا
    text = re.sub(r"ى", "ي", text)                # ى -> ي
    text = re.sub(r"ة", "ه", text)                # ة -> ه
    text = re.sub(r"[\u064B-\u0652]", "", text)   # remove tashkeel
    text = re.sub(r"ـ", "", text)                 # التوصيـــل -> التوصيل
    text = re.sub(r"(.)\1{2,}", r"\1", text)      # حلوووو -> حلو
    return text.strip()

df["clean_text"] = df["content"].apply(normalize_arabic)
df[["content", "clean_text"]].sample(10)

,content,clean_text
45340,خدمة العملاء جدا ممتازه,خدمه العملاء جدا ممتازه
60781,سيئ جدا. طلب عشاء . لي ٢ساعتين ماوصل لاحد الآن...,سيئ جدا. طلب عشاء . لي ٢ساعتين ماوصل لاحد الان...
30440,البرنامج جيد من ناحية العروض للمستخدم الجديد. ...,البرنامج جيد من ناحيه العروض للمستخدم الجديد. ...
6617,سيء جدا اشتركت معهم سنة توصيل مجاني و يوم وصل ...,سيء جدا اشتركت معهم سنه توصيل مجاني و يوم وصل ...
49979,تجربة مميزة جدا,تجربه مميزه جدا
48774,ممتاز جدا جدا الله يوفقكم ويوسع رزقكم,ممتاز جدا جدا الله يوفقكم ويوسع رزقكم
6333,التطبيق ماعاد يشتغل على جهازي صاير سيء و يعلق,التطبيق ماعاد يشتغل علي جهازي صاير سيء و يعلق
8559,السلام عليكم أتمنى وضع خيار للمظهر الداكن,السلام عليكم اتمني وضع خيار للمظهر الداكن
10589,كل ما اطلب يقول عذرا وانا عندي رصيد ف المحفظة ...,كل ما اطلب يقول عذرا وانا عندي رصيد ف المحفظه ...
9545,طلال نواف المطيري,طلال نواف المطيري


In [6]:
df.to_csv("data/clean/reviews_clean.csv", index=False, encoding="utf-8-sig")
print("Saved with clean_text!", df.shape)

Saved with clean_text! (46148, 11)


In [7]:
import re
from collections import Counter

# Common words that don't tell us anything (we ignore them)
stopwords = set("""
في من الي علي عن ما لا لم لن ان انا انت هو هي هذا هذه ذا كل مع او ثم قد كان
مره جدا يا الله والله التطبيق البرنامج برنامج تطبيق شي اللي عشان بس وش ليش
""".split())

# Take reviews with 1, 2 or 3 stars (where complaints usually are)
complaints = df[df["score"] <= 3]
print("Reviews with 1-3 stars:", len(complaints))

# Extract Arabic words, remove stopwords and very short words
words = []
for text in complaints["clean_text"]:
    for w in re.findall(r"[\u0600-\u06FF]+", text):
        if w not in stopwords and len(w) > 2:
            words.append(w)

top = pd.DataFrame(Counter(words).most_common(50), columns=["word", "count"])
top

Reviews with 1-3 stars: 24851


,word,count
0,الطلب,6905
1,خدمه,4137
2,ولا,4059
3,التوصيل,3448
4,العملاء,2974
5,ريال,2724
6,توصيل,2327
7,سيء,2309
8,طلب,2243
9,غير,2118


In [8]:
topics = {
    "Delay":            ["تاخير", "تاخر", "متاخر", "ساعه", "ساعتين", "ماوصل", "ما وصل", "انتظار", "انتظرت"],
    "Price":            ["غالي", "اسعار", "سعر", "رسوم", "مبالغ", "الغلا"],
    "Driver":           ["مندوب", "السائق", "سائق", "كابتن"],
    "Order issue":      ["ناقص", "غلط", "بارد", "خربان", "مو نفس"],
    "Cancellation":     ["الغاء", "الغي", "انلغي", "ملغي", "كنسل"],
    "Restaurant":       ["مطعم", "مطاعم"],
    "Payment & refund": ["الدفع", "استرجاع", "استرداد", "فلوسي", "المبلغ", "المحفظه", "رصيد", "تعويض", "انخصم"],
    "Customer service": ["خدمه العملاء", "الدعم", "الشات", "محادثه", "ما يردون", "مايردون"],
    "App technical":    ["يعلق", "تعليق", "ما يفتح", "مايفتح", "تحديث", "تسجيل", "خلل"],
    "Offers":           ["عروض", "كوبون", "مجاني", "اشتراك", "خصم"],
}

# Create one True/False column for each topic
for topic, keywords in topics.items():
    df[topic] = df["clean_text"].apply(lambda t: any(k in t for k in keywords))

topic_cols = list(topics.keys())
negative = df[df["score"] <= 2]

# How many negative reviews got at least one topic?
coverage = negative[topic_cols].any(axis=1).mean() * 100
print(f"Negative reviews with at least one topic: {coverage:.1f}%")
print("-" * 40)

# % of negative reviews mentioning each topic
(negative[topic_cols].mean() * 100).round(1).sort_values(ascending=False)

Negative reviews with at least one topic: 61.9%
----------------------------------------


Price               15.6
Customer service    15.5
Payment & refund    13.9
Delay               13.4
Driver              11.2
Restaurant          10.2
Offers               9.7
Order issue          9.5
Cancellation         6.6
App technical        4.5
dtype: float64

In [9]:
pd.set_option("display.max_colwidth", 150)   # show longer text in tables

has_topic = negative[topic_cols].any(axis=1)
uncategorized = negative[~has_topic]
categorized = negative[has_topic]

print("Uncategorized negative reviews:", len(uncategorized))
print("Average length - categorized:  ", round(categorized["clean_text"].str.len().mean()), "characters")
print("Average length - uncategorized:", round(uncategorized["clean_text"].str.len().mean()), "characters")

uncategorized[["score", "content"]].sample(20)

Uncategorized negative reviews: 8994
Average length - categorized:   141 characters
Average length - uncategorized: 48 characters


,score,content
29386,1,سيء و الخدمة بإنحدااار
12460,1,تطبيق سيئ ، لاتقدير لعملائهم القدماء ولااحترام للتواصل معهم ، مشاكل كثيرة ولاحلول أبدا . لقد تعرضت لثلاث مشاكل ولم تحل اي واحده فيهم .
20523,1,حسبي الله ونعم الوكيل ع صاحب تطبيق جاهز هو وخدمة التواصل كلمتهم وراسلتهم ولا يتجاوبون لكن مااقول الا الله لا يبارك لهم ولا يوفقهم وحسبي الله ونعم ...
60114,1,برنامج سي جدن حرميه كل شي سيئه جدا في هذا البرنامج
61420,1,اسو تطبيق واسو خدمه ونجمه كثيره عليه... 🤬🤬🤬
59364,1,بقت مقرفه
1617,1,التطبيق لايعمل على التابلت اذا جيت اسوي طلب يقول غير قادر على معالجة طلبك !
3824,1,يمنيا بالله من اسوا تطبيقات التوصيل
22134,1,نجمة وحدة وكثيرة عليك...
8576,1,برنامج خايس


In [10]:
# Remove reviews clearly from Egypt (currency or "in Egypt")
egypt = df["clean_text"].str.contains("جنيه|في مصر")
print("Egypt reviews removed:", egypt.sum())
df = df[~egypt].copy()

# Improved topics (new keywords added + new "Coverage area" topic)
topics = {
    "Delay":            ["تاخير", "تاخر", "متاخر", "ساعه", "ساعتين", "ماوصل", "ما وصل", "انتظار", "انتظرت",
                         "دقايق", "دقائق", "الوقت المحدد", "مده التوصيل"],
    "Price":            ["غالي", "اسعار", "سعر", "رسوم", "مبالغ", "الغلا"],
    "Driver":           ["مندوب", "السائق", "سائق", "كابتن"],
    "Order issue":      ["ناقص", "غلط", "بارد", "خربان", "مو نفس"],
    "Cancellation":     ["الغاء", "الغي", "انلغي", "ملغي", "كنسل"],
    "Restaurant":       ["مطعم", "مطاعم"],
    "Payment & refund": ["الدفع", "استرجاع", "استرداد", "فلوسي", "المبلغ", "المحفظه", "رصيد", "تعويض", "انخصم",
                         "نصب", "نصابين", "حراميه", "حرميه", "سرقه"],
    "Customer service": ["خدمه العملاء", "الدعم", "الشات", "محادثه", "ما يردون", "مايردون",
                         "التواصل", "يتجاوبون", "راسلتهم", "كلمتهم"],
    "App technical":    ["يعلق", "تعليق", "ما يفتح", "مايفتح", "تحديث", "تسجيل", "خلل",
                         "لايعمل", "لا يعمل", "ما يشتغل", "مايشتغل"],
    "Offers":           ["عروض", "كوبون", "مجاني", "اشتراك", "خصم"],
    "Coverage area":    ["مناطق", "منطقتي", "حينا", "المسافه", "مافي توصيل"],
}

for topic, keywords in topics.items():
    df[topic] = df["clean_text"].apply(lambda t: any(k in t for k in keywords))

topic_cols = list(topics.keys())
negative = df[df["score"] <= 2]

coverage = negative[topic_cols].any(axis=1).mean() * 100
print(f"Negative reviews with at least one topic: {coverage:.1f}%")
print("-" * 40)
(negative[topic_cols].mean() * 100).round(1).sort_values(ascending=False)

Egypt reviews removed: 123
Negative reviews with at least one topic: 64.8%
----------------------------------------


Payment & refund    18.3
Customer service    16.8
Price               15.6
Delay               13.8
Driver              11.1
Restaurant          10.2
Offers               9.7
Order issue          9.5
Cancellation         6.6
App technical        5.3
Coverage area        0.7
dtype: float64

In [11]:
# How many negative reviews does each keyword match?
for k in topics["Payment & refund"]:
    n = negative["clean_text"].str.contains(k).sum()
    print(f"{k:<10} {n}")

الدفع      822
استرجاع    156
استرداد    230
فلوسي      430
المبلغ     956
المحفظه    335
رصيد       224
تعويض      869
انخصم      12
نصب        484
نصابين     485
حراميه     256
حرميه      38
سرقه       144


In [12]:
# Payment & refund: keep only money-related words
topics["Payment & refund"] = ["الدفع", "استرجاع", "استرداد", "فلوسي", "المبلغ",
                              "المحفظه", "رصيد", "تعويض", "انخصم"]

# New topic: customers accusing the app of fraud/theft
topics["Fraud accusations"] = ["نصب", "نصابين", "حرامي", "حرميه", "سرق", "احتيال"]

# Remove the tiny topic (0.7%)
topics.pop("Coverage area")
df = df.drop(columns=["Coverage area"])

# Re-apply all topics
for topic, keywords in topics.items():
    df[topic] = df["clean_text"].apply(lambda t: any(k in t for k in keywords))

topic_cols = list(topics.keys())
negative = df[df["score"] <= 2]

coverage = negative[topic_cols].any(axis=1).mean() * 100
print(f"Negative reviews with at least one topic: {coverage:.1f}%")

# Save the final classified data
df.to_csv("data/clean/reviews_clean.csv", index=False, encoding="utf-8-sig")
print("Saved!", df.shape)
print("-" * 40)
(negative[topic_cols].mean() * 100).round(1).sort_values(ascending=False)

Negative reviews with at least one topic: 65.0%
Saved! (46025, 22)
----------------------------------------


Customer service     16.8
Price                15.6
Payment & refund     14.0
Delay                13.8
Driver               11.1
Restaurant           10.2
Offers                9.7
Order issue           9.5
Fraud accusations     6.9
Cancellation          6.6
App technical         5.3
dtype: float64

In [13]:
# Make a readable column with the topics of each review
df["topics_list"] = df[topic_cols].apply(
    lambda row: ", ".join([t for t in topic_cols if row[t]]), axis=1
)

# Take 30 random negative reviews that got at least one topic
check = df[(df["score"] <= 2) & (df["topics_list"] != "")].sample(30, random_state=42)
check[["content", "topics_list"]]

,content,topics_list
10170,ما عندهم او شيء شبكه وتعاملهم سيء يجيبوا لك متاخر,Delay
52803,طلبت من السوبر ماركت واتصل ع المرسول مايرد علينا أقولوا رد ولانكنسل الطلب قلي هذا تهديد المهم ماكثرت اشتكيتك الدعم مرتين ماردو عليا ولما مااستلمنا...,"Cancellation, Customer service"
2856,سعر التوصيل كنك طلبت طلب ثاني,Price
7113,التطبيق فيه مشكله مع الراجحي كل شوي يرفض ويلغي الدفع الالكتروني .. الان حاط بطاقتي غير متاحه ... بدون سبب .. وقبلها ماكان يقبل .. وكثيره مشاكل بال...,Payment & refund
28711,سيى جدا وتعاملهم سي وولم يرجعون لي فلوسي حسبي الله ونعم الوكيل,Payment & refund
29482,مع الاسف طلبت من مطعم وتاخر علي ساعة و٤٠ د واتضح بعد ذلك ان فيه مندوب كيتا اخر استلم طلبي بالخطا وتم الغاء طلبي ، هاذي اخر مرة اطلب منكم,"Delay, Driver, Cancellation, Restaurant"
19096,تطبيق زباله ماعنده دعم للمندوب وينصب ع المندوب خصميات ولا يستطيع التواصل معهم/ اسوء تطبيق اللي يشتغل فيه يظحك على نفسه ويكذبون بالمواقع يظهر لك ٥ ...,"Driver, Customer service, Offers, Fraud accusations"
54222,سيء جدا ولا خدوم اريد افعل التطبيق نادي لي شهر ولا اعرف اتواصل مع مندوب يفيدني في شيء,Driver
10298,قيمة اي وجبة من اي مطعم بزيادة ٦ريال نصابين,"Restaurant, Fraud accusations"
15116,سيء جداً لم احصل على خدمة جيدة أبداً لقد تأخر الطلب كثيراً ولم أجد رد من خدمة العملاء و فوق ذلك بعد وصول الطعام وصل بصورة مبعثرة جداً,"Delay, Customer service"


In [14]:
check_ids = check.index   # remember the same 30 reviews to compare

topics["Driver"] = ["مندوب", "مناديب", "السائق", "سائق", "سايق", "كابتن", "المرسول", "موصل"]
topics["Order issue"] = ["ناقص", "غلط", "بارد", "خربان", "مو نفس", "اخطاء", "مبعثر", "انكب", "مكبوب"]
topics["Cancellation"] = ["الغاء الطلب", "الغاء طلبي", "الغو الطلب", "الغو طلبي", "لغو طلبي", "لغو الطلب",
                          "انلغي", "يتلغي", "ملغي", "كنسل الطلب", "كنسلو", "تم الغاء"]
topics["Customer service"] = ["خدمه العملاء", "الدعم", "الشات", "محادثه", "ما يردون", "مايردون",
                              "تواصل", "يتجاوبون", "راسلتهم", "كلمتهم"]
topics["Fraud accusations"] = ["نصب", "نصاب", "حرامي", "حرميه", "سرق", "احتيال"]

# Remove Restaurant (mentioning a restaurant is not a complaint about it)
topics.pop("Restaurant")
df = df.drop(columns=["Restaurant"])

def has_topic(text, keywords):
    # "after it arrived" is not a delay
    text = text.replace("بعد ما وصل", "").replace("بعد ماوصل", "")
    return any(k in text for k in keywords)

for topic, keywords in topics.items():
    df[topic] = df["clean_text"].apply(lambda t: has_topic(t, keywords))

topic_cols = list(topics.keys())
df["topics_list"] = df[topic_cols].apply(
    lambda row: ", ".join([t for t in topic_cols if row[t]]), axis=1
)

df.loc[check_ids, ["content", "topics_list"]]

,content,topics_list
10170,ما عندهم او شيء شبكه وتعاملهم سيء يجيبوا لك متاخر,Delay
52803,طلبت من السوبر ماركت واتصل ع المرسول مايرد علينا أقولوا رد ولانكنسل الطلب قلي هذا تهديد المهم ماكثرت اشتكيتك الدعم مرتين ماردو عليا ولما مااستلمنا...,"Driver, Cancellation, Customer service"
2856,سعر التوصيل كنك طلبت طلب ثاني,Price
7113,التطبيق فيه مشكله مع الراجحي كل شوي يرفض ويلغي الدفع الالكتروني .. الان حاط بطاقتي غير متاحه ... بدون سبب .. وقبلها ماكان يقبل .. وكثيره مشاكل بال...,Payment & refund
28711,سيى جدا وتعاملهم سي وولم يرجعون لي فلوسي حسبي الله ونعم الوكيل,Payment & refund
29482,مع الاسف طلبت من مطعم وتاخر علي ساعة و٤٠ د واتضح بعد ذلك ان فيه مندوب كيتا اخر استلم طلبي بالخطا وتم الغاء طلبي ، هاذي اخر مرة اطلب منكم,"Delay, Driver, Cancellation"
19096,تطبيق زباله ماعنده دعم للمندوب وينصب ع المندوب خصميات ولا يستطيع التواصل معهم/ اسوء تطبيق اللي يشتغل فيه يظحك على نفسه ويكذبون بالمواقع يظهر لك ٥ ...,"Driver, Customer service, Offers, Fraud accusations"
54222,سيء جدا ولا خدوم اريد افعل التطبيق نادي لي شهر ولا اعرف اتواصل مع مندوب يفيدني في شيء,"Driver, Customer service"
10298,قيمة اي وجبة من اي مطعم بزيادة ٦ريال نصابين,Fraud accusations
15116,سيء جداً لم احصل على خدمة جيدة أبداً لقد تأخر الطلب كثيراً ولم أجد رد من خدمة العملاء و فوق ذلك بعد وصول الطعام وصل بصورة مبعثرة جداً,"Delay, Order issue, Customer service"


In [15]:
# Check the full text of review 3887 (it was cut in the table)
print(df.loc[3887, "content"])
print("=" * 40)

# Add the missing cancellation word
topics["Cancellation"].append("ينلغي")
df["Cancellation"] = df["clean_text"].apply(lambda t: has_topic(t, topics["Cancellation"]))
df["topics_list"] = df[topic_cols].apply(
    lambda row: ", ".join([t for t in topic_cols if row[t]]), axis=1
)

# Final numbers
negative = df[df["score"] <= 2]
coverage = negative[topic_cols].any(axis=1).mean() * 100
print(f"Negative reviews with at least one topic: {coverage:.1f}%")

# Save final data
df.to_csv("data/clean/reviews_clean.csv", index=False, encoding="utf-8-sig")
print("Saved!", df.shape)
print("-" * 40)
(negative[topic_cols].mean() * 100).round(1).sort_values(ascending=False)

تجربتي بها الجيد و السيئ علما بأنني مشترك في خدمة التوصيل المجاني وادفع رسوم اغلب المشاكل الذي أصادفها من السائقين و أخطاء في الطلب وكأنهم يتعمدون ذلك في الاخير حصل خطأ في توصيل طلب وتأخر كثيرا حتى وصل وكان قد برد الأكل وطلبت استبداله ولم يستجيب أحد فطلبت إرجاع قيمة الطلب وقد تعهدوا بذلك والى اليوم لم يرجع لي ذلك المبلغ واليوم أخر يوم يكون هذا البرنامج في جهازي ✋️
Negative reviews with at least one topic: 65.4%
Saved! (46025, 22)
----------------------------------------


Customer service     19.6
Price                15.6
Driver               14.3
Payment & refund     14.0
Delay                13.8
Order issue          10.1
Offers                9.7
Fraud accusations     7.6
App technical         5.3
Cancellation          3.9
dtype: float64